In [1]:
# ==============================================================================
# CELDA 1 (NOTEBOOK DENUNCIAS): LIBRERÍAS + INGESTA 2016-2017
# ==============================================================================
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

RUTA_DENUNCIAS = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/DENUNCIAS/'

archivos_denuncias = {
    2016: 'DENUNCIAS_2016_20240729_PUBL.csv',
    2017: 'DENUNCIAS_2017_20240729_PUBL.csv',
}

denuncias_brutas = {}
for anio, nombre in archivos_denuncias.items():
    ruta = RUTA_DENUNCIAS + nombre
    for enc in ['utf-8-sig', 'latin-1', 'cp1252']:
        try:
            denuncias_brutas[anio] = pd.read_csv(ruta, encoding=enc, sep=';', low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    print(f"OK | {anio} | {denuncias_brutas[anio].shape[0]} filas x {denuncias_brutas[anio].shape[1]} columnas")

print("\nColumnas 2016:", denuncias_brutas[2016].columns.tolist())

Mounted at /content/drive
OK | 2016 | 11502 filas x 33 columnas
OK | 2017 | 12672 filas x 34 columnas

Columnas 2016: ['AGNO', 'DEN_ID', 'DEN_CANAL', 'DEN_ESTADO', 'DEN_FEC_CREACION', 'DEN_MES_CREACION', 'DEN_TRIMESTRE_CREACION', 'DEN_FEC_TERMINO_DENUNCIA', 'DEN_OFICINA', 'DEN_DEPARTAMENTO', 'DEN_AMBITO', 'DEN_TEMA', 'DEN_SUBTEMA', 'FIS_ID', 'FIS_ID_SEGUIMIENTO', 'PA_ID', 'DEN_MRUN', 'DEN_AFECTADO', 'DEN_SEXO', 'DEN_TIPO', 'AFEC_MRUN', 'AFEC_SEXO', 'RBD', 'EE_NOMBRE', 'EE_COD_REGION', 'EE_COD_PROVINCIA', 'EE_COD_COMUNA', 'EE_NOM_COMUNA', 'EE_COD_DEPE', 'EE_DEPE_AGRUP', 'AFEC_COD_ENSE2', 'DEN_REGION', 'DEN_CIBERBULLYING']


In [2]:
# ==============================================================================
# CELDA 2: AGREGACIÓN DE DENUNCIAS POR RBD (2016-2017 -> features por colegio)
# ==============================================================================
denuncias_todas = pd.concat(denuncias_brutas.values(), ignore_index=True)

# RBD blindado: solo denuncias asociadas a un establecimiento con RBD válido
denuncias_todas['RBD'] = pd.to_numeric(denuncias_todas['RBD'], errors='coerce')
denuncias_todas = denuncias_todas.dropna(subset=['RBD'])
denuncias_todas['RBD'] = denuncias_todas['RBD'].astype('Int64').astype(str)

agg = denuncias_todas.groupby('RBD').agg(
    denuncias_total=('DEN_ID', 'count'),
    denuncias_resueltas=('DEN_ESTADO', lambda x: (x == 1).sum()),
    denuncias_fiscalizacion=('DEN_DEPARTAMENTO', lambda x: (x == 2).sum()),
    denuncias_juridica=('DEN_DEPARTAMENTO', lambda x: (x == 3).sum()),
    denuncias_ciberbullying=('DEN_CIBERBULLYING', lambda x: (x == 1).sum()),
).reset_index().rename(columns={'RBD': 'rbd'})

print(f"Colegios con al menos 1 denuncia (2016-17): {len(agg)}")
print(agg.describe())

RUTA_SALIDA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
agg.to_parquet(RUTA_SALIDA + 'denuncias_2016_17_por_rbd.parquet', index=False)
print(f"\nGuardado: {RUTA_SALIDA}denuncias_2016_17_por_rbd.parquet")

Colegios con al menos 1 denuncia (2016-17): 6230
       denuncias_total  denuncias_resueltas  denuncias_fiscalizacion  \
count      6230.000000          6230.000000              6230.000000   
mean          3.717657             3.717657                 0.383467   
std           3.954598             3.954598                 0.859620   
min           1.000000             1.000000                 0.000000   
25%           1.000000             1.000000                 0.000000   
50%           2.000000             2.000000                 0.000000   
75%           5.000000             5.000000                 1.000000   
max          84.000000            84.000000                13.000000   

       denuncias_juridica  denuncias_ciberbullying  
count         6230.000000                   6230.0  
mean             0.719743                      0.0  
std              1.198321                      0.0  
min              0.000000                      0.0  
25%              0.000000            

In [3]:
# ==============================================================================
# VERIFICACIÓN RÁPIDA
# ==============================================================================
print(denuncias_todas['DEN_ESTADO'].value_counts(dropna=False))
print(denuncias_todas['DEN_CIBERBULLYING'].value_counts(dropna=False))
print(denuncias_todas['DEN_CIBERBULLYING'].dtype)

DEN_ESTADO
1    23161
Name: count, dtype: int64
DEN_CIBERBULLYING
     17978
0     4720
1      463
Name: count, dtype: int64
object


In [4]:
# ==============================================================================
# CELDA 2 CORREGIDA: AGREGACIÓN DE DENUNCIAS POR RBD
# ==============================================================================
denuncias_todas = pd.concat(denuncias_brutas.values(), ignore_index=True)

denuncias_todas['RBD'] = pd.to_numeric(denuncias_todas['RBD'], errors='coerce')
denuncias_todas = denuncias_todas.dropna(subset=['RBD'])
denuncias_todas['RBD'] = denuncias_todas['RBD'].astype('Int64').astype(str)

agg = denuncias_todas.groupby('RBD').agg(
    denuncias_total=('DEN_ID', 'count'),
    denuncias_fiscalizacion=('DEN_DEPARTAMENTO', lambda x: (x == 2).sum()),
    denuncias_juridica=('DEN_DEPARTAMENTO', lambda x: (x == 3).sum()),
    denuncias_ciberbullying=('DEN_CIBERBULLYING', lambda x: (x.astype(str) == '1').sum()),
).reset_index().rename(columns={'RBD': 'rbd'})

print(f"Colegios con al menos 1 denuncia: {len(agg)}")
print(agg.describe())

RUTA_SALIDA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
agg.to_parquet(RUTA_SALIDA + 'denuncias_2016_17_por_rbd.parquet', index=False)
print("Guardado OK")

Colegios con al menos 1 denuncia: 6230
       denuncias_total  denuncias_fiscalizacion  denuncias_juridica  \
count      6230.000000              6230.000000         6230.000000   
mean          3.717657                 0.383467            0.719743   
std           3.954598                 0.859620            1.198321   
min           1.000000                 0.000000            0.000000   
25%           1.000000                 0.000000            0.000000   
50%           2.000000                 0.000000            0.000000   
75%           5.000000                 1.000000            1.000000   
max          84.000000                13.000000           17.000000   

       denuncias_ciberbullying  
count              6230.000000  
mean                  0.074318  
std                   0.282352  
min                   0.000000  
25%                   0.000000  
50%                   0.000000  
75%                   0.000000  
max                   3.000000  
Guardado OK


In [5]:
# ==============================================================================
# CELDA 3: INTEGRAR DENUNCIAS A LA TABLA DE ENTRENAMIENTO
# ==============================================================================
RUTA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
df_modelo = pd.read_parquet(RUTA + 'tabla_modelo_final_v2.parquet')
denuncias = pd.read_parquet(RUTA + 'denuncias_2016_17_por_rbd.parquet')

df_modelo_v3 = pd.merge(df_modelo, denuncias, on='rbd', how='left', validate='one_to_one')

# Colegios sin denuncia = 0 (no nulo, es ausencia real de registro)
cols_den = [c for c in denuncias.columns if c != 'rbd']
df_modelo_v3[cols_den] = df_modelo_v3[cols_den].fillna(0)

print(f"Filas: {len(df_modelo_v3)} (antes: {len(df_modelo)})")
print(f"Con al menos 1 denuncia: {(df_modelo_v3['denuncias_total']>0).sum()}")

df_modelo_v3.to_parquet(RUTA + 'tabla_modelo_final_v3.parquet', index=False)
print(df_modelo_v3.shape)

Filas: 7754 (antes: 7754)
Con al menos 1 denuncia: 4609
(7754, 44)


In [6]:
# ==============================================================================
# CELDA 4: DENUNCIAS 2018-2022 (para el set de validación bienio 2023-24)
# ==============================================================================
archivos_v2 = {
    2018: 'DENUNCIAS_2018_20240729_PUBL.csv',
    2019: 'DENUNCIAS_2019_20240729_PUBL.csv',
    2020: 'DENUNCIAS_2020_20240729_PUBL.csv',
    2021: 'DENUNCIAS_2021_20240729_PUBL.csv',
    2022: 'DENUNCIAS_2022_PUBL.csv',
}

den_brutas_v2 = {}
for anio, nombre in archivos_v2.items():
    ruta = RUTA_DENUNCIAS + nombre
    for enc in ['utf-8-sig', 'latin-1', 'cp1252']:
        try:
            den_brutas_v2[anio] = pd.read_csv(ruta, encoding=enc, sep=';', low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    print(f"OK | {anio} | {den_brutas_v2[anio].shape[0]} filas x {den_brutas_v2[anio].shape[1]} columnas")

den_todas_v2 = pd.concat(den_brutas_v2.values(), ignore_index=True)
den_todas_v2['RBD'] = pd.to_numeric(den_todas_v2['RBD'], errors='coerce')
den_todas_v2 = den_todas_v2.dropna(subset=['RBD'])
den_todas_v2['RBD'] = den_todas_v2['RBD'].astype('Int64').astype(str)

agg_v2 = den_todas_v2.groupby('RBD').agg(
    denuncias_total=('DEN_ID', 'count'),
    denuncias_fiscalizacion=('DEN_DEPARTAMENTO', lambda x: (x == 2).sum()),
    denuncias_juridica=('DEN_DEPARTAMENTO', lambda x: (x == 3).sum()),
    denuncias_ciberbullying=('DEN_CIBERBULLYING', lambda x: (x.astype(str) == '1').sum()),
).reset_index().rename(columns={'RBD': 'rbd'})

print(f"Colegios con denuncia (2018-22): {len(agg_v2)}")
RUTA_SALIDA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
agg_v2.to_parquet(RUTA_SALIDA + 'denuncias_2018_22_por_rbd.parquet', index=False)
print("Guardado OK")

OK | 2018 | 15017 filas x 34 columnas
OK | 2019 | 12016 filas x 34 columnas
OK | 2020 | 3379 filas x 34 columnas
OK | 2021 | 3961 filas x 35 columnas
OK | 2022 | 16161 filas x 42 columnas
Colegios con denuncia (2018-22): 8083
Guardado OK
